In [0]:
print("Commodity Closing Simulation - Databricks project just started")
print("We're ready!")

Commodity Closing Simulation - Databricks project just started
We're ready!


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import datetime

print("Libraries uploaded")

Libraries uploaded


In [0]:
# Creating sample Commodity (Energy) trading data
data = [
    ("2025-01-05", "Power", "DE", 85.40, 1200, "Buy"),
    ("2025-01-05", "Gas", "NL", 32.10, 5000, "Sell"),
    ("2025-01-12", "Power", "DE", 91.20, 800, "Buy"),
    ("2025-01-18", "CO2", "EU", 68.50, 2000, "Buy"),
    ("2025-01-22", "Gas", "NL", 29.80, 4500, "Sell"),
    ("2025-01-28", "Power", "DE", 78.90, 1500, "Sell"),
    ("2025-02-03", "Power", "DE", 88.30, 1100, "Buy"),
    ("2025-02-10", "CO2", "EU", 71.20, 1800, "Sell"),
    ("2025-02-15", "Gas", "NL", 31.50, 5200, "Buy"),
    ("2025-02-25", "Power", "DE", 82.70, 950, "Sell"),
]

columns = ["trade_date", "commodity", "market", "price", "volume", "side"]

df = spark.createDataFrame(data, columns)

# Convert trade_date to proper date format
df = df.withColumn("trade_date", to_date(col("trade_date")))

print("Sample commodity data created successfully")
df.show()

Sample commodity data created successfully
+----------+---------+------+-----+------+----+
|trade_date|commodity|market|price|volume|side|
+----------+---------+------+-----+------+----+
|2025-01-05|    Power|    DE| 85.4|  1200| Buy|
|2025-01-05|      Gas|    NL| 32.1|  5000|Sell|
|2025-01-12|    Power|    DE| 91.2|   800| Buy|
|2025-01-18|      CO2|    EU| 68.5|  2000| Buy|
|2025-01-22|      Gas|    NL| 29.8|  4500|Sell|
|2025-01-28|    Power|    DE| 78.9|  1500|Sell|
|2025-02-03|    Power|    DE| 88.3|  1100| Buy|
|2025-02-10|      CO2|    EU| 71.2|  1800|Sell|
|2025-02-15|      Gas|    NL| 31.5|  5200| Buy|
|2025-02-25|    Power|    DE| 82.7|   950|Sell|
+----------+---------+------+-----+------+----+



In [0]:
# Save the data as a Delta Lake table
df.write.format("delta").mode("overwrite").saveAsTable("commodity_trades")

print("Data successfully saved as Delta table: commodity_trades")

Data successfully saved as Delta table: commodity_trades


In [0]:
# Create Monthly Closing Summary
monthly_summary = spark.sql("""
    SELECT 
        date_format(trade_date, 'yyyy-MM') AS month,
        commodity,
        market,
        ROUND(AVG(price), 2) AS avg_price,
        SUM(volume) AS total_volume,
        COUNT(*) AS number_of_trades,
        SUM(CASE WHEN side = 'Buy' THEN volume ELSE 0 END) AS buy_volume,
        SUM(CASE WHEN side = 'Sell' THEN volume ELSE 0 END) AS sell_volume
    FROM commodity_trades
    GROUP BY date_format(trade_date, 'yyyy-MM'), commodity, market
    ORDER BY month, commodity
""")

print("Monthly Closing Summary:")
monthly_summary.show()

Monthly Closing Summary:
+-------+---------+------+---------+------------+----------------+----------+-----------+
|  month|commodity|market|avg_price|total_volume|number_of_trades|buy_volume|sell_volume|
+-------+---------+------+---------+------------+----------------+----------+-----------+
|2025-01|      CO2|    EU|     68.5|        2000|               1|      2000|          0|
|2025-01|      Gas|    NL|    30.95|        9500|               2|         0|       9500|
|2025-01|    Power|    DE|    85.17|        3500|               3|      2000|       1500|
|2025-02|      CO2|    EU|     71.2|        1800|               1|         0|       1800|
|2025-02|      Gas|    NL|     31.5|        5200|               1|      5200|          0|
|2025-02|    Power|    DE|     85.5|        2050|               2|      1100|        950|
+-------+---------+------+---------+------------+----------------+----------+-----------+



In [0]:
# Simple PnL calculation (for demonstration purposes)
# Assuming Buy = cost, Sell = revenue (very simplified)

pnl_df = spark.sql("""
    SELECT 
        commodity,
        market,
        ROUND(SUM(CASE WHEN side = 'Sell' THEN price * volume ELSE 0 END), 2) AS total_revenue,
        ROUND(SUM(CASE WHEN side = 'Buy' THEN price * volume ELSE 0 END), 2) AS total_cost,
        ROUND(
            SUM(CASE WHEN side = 'Sell' THEN price * volume ELSE 0 END) -
            SUM(CASE WHEN side = 'Buy' THEN price * volume ELSE 0 END)
        , 2) AS simple_pnl
    FROM commodity_trades
    GROUP BY commodity, market
    ORDER BY commodity
""")

print("Simple PnL by Commodity & Market:")
pnl_df.show()

Simple PnL by Commodity & Market:
+---------+------+-------------+----------+----------+
|commodity|market|total_revenue|total_cost|simple_pnl|
+---------+------+-------------+----------+----------+
|      CO2|    EU|     128160.0|  137000.0|   -8840.0|
|      Gas|    NL|     294600.0|  163800.0|  130800.0|
|    Power|    DE|     196915.0|  272570.0|  -75655.0|
+---------+------+-------------+----------+----------+



In [0]:
# Verify the Delta table
print("Tables in the catalog:")
spark.sql("SHOW TABLES").show()

print("\nSchema of commodity_trades table:")
spark.sql("DESCRIBE commodity_trades").show()

Tables in the catalog:
+--------+----------------+-----------+
|database|       tableName|isTemporary|
+--------+----------------+-----------+
| default|commodity_trades|      false|
+--------+----------------+-----------+


Schema of commodity_trades table:
+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|trade_date|     date|   NULL|
| commodity|   string|   NULL|
|    market|   string|   NULL|
|     price|   double|   NULL|
|    volume|   bigint|   NULL|
|      side|   string|   NULL|
+----------+---------+-------+



In [0]:
print("="*60)
print("Commodity Closing Simulation - Completed Successfully")
print("="*60)
print("What we did:")
print("1. Created sample commodity trading data")
print("2. Saved it as a Delta Lake table")
print("3. Generated Monthly Closing Summary")
print("4. Calculated simple PnL")
print("="*60)

Commodity Closing Simulation - Completed Successfully
What we did:
1. Created sample commodity trading data
2. Saved it as a Delta Lake table
3. Generated Monthly Closing Summary
4. Calculated simple PnL
